# TFM - Predicción de cancelaciones - Uso del modelo


Valores permitidos de la variables:

**LLEGADA** ('AAAA-MM-DD', ej: '2023-11-06')

**NOCHES** (entero, ej: 17)

**PAX** (entero, ej: 2)

**ADULTOS** (entero, ej: 2)

**CLIENTE** ('MOTOR RESERVAS GLOBAL', 'MOTOR RESERVAS GLOBAL - DE', 'AUREA TRAVEL CLUB_SOCIOS', 'CALL CENTER', 'OTA EUROPE BV',
'OTA LATAM S.A.', 'HOTEL COLLECT_OTA AMERICAS INC', 'OTA AMERICAS INC', ...)

**FECHA_TOMA** ('AAAA-MM-DD hh:mm:ss', ej '2023-02-22 12:48:29')

**FIDELIDAD** (None, 'Aurea Rewards', Aurea Connect', 'Partner Rewards')

**PAIS** ('ESTADOS UNIDOS', 'CANADA', 'MEXICO', 'ARGENTINA', 'SIN PAIS', 'ESPAÑA', 'REPUBLICA DOMINICANA', 'REINO UNIDO', 'ALEMANIA', 'URUGUAY', 'DOMINICA', ...)

**SEGMENTO** ('Fixed Rates', 'BAR', 'Loyalty', 'Package', 'Group Leisure', 'Fixed rates', 'Weddings', ...)

**FUENTE_NEGOCIO** ('T.O. / T.A.', 'DIRECT SALES', 'E-COMMERCE', 'CORPORATE', 'OTHERS')

**NOMBRE_HOTEL** ('Complejo Punta Cana', 'Complejo Costa Mujeres', 'Complejo Riviera Maya')

**NOMBRE_HABITACION** ('OTROS', 'CMU JUNIOR SUITE PS', 'BAV SUPERIOR JS GV', 'CMU JUNIOR SUITE GV', 'COL DELUXE GARDEN VIEW', 'SIG JUNIOR SUITE GV', 'PC DELUXE GARDEN VIEW', 'SIG JUNIOR SUITE PS', 'WS JUNIOR SUITE GV', 'PC DELUXE POOLSIDE', 'SIGC JUNIOR SUITE GV', 'KAN DELUXE GARDEN VIEW', 'SIG JUNIOR SUITE GV/PV', 'PC JUNIOR SUITE GV', 'BAV JUNIOR SUITE GV', 'SIG JS JACUZZI TERR PS', 'CMU FS JUNIOR SUITE BS', ...)

**VALOR_RESERVA** (float, ej 4899.40)


(Ver sección '*Valores permitidos categóricas*' para todos los valores permitidos de las variables categóricas: )

*(Example booking input: see `get_features()` arguments below.)*

**ejemplo 1:**

LLEGADA:	2023-11-06

NOCHES:	17

PAX:	2

ADULTOS:	2

CLIENTE:	AUREA TRAVEL CLUB_SOCIOS

FECHA_TOMA:	2023-02-22 12:48:29

FIDELIDAD:	None

PAIS:	ESTADOS UNIDOS

SEGMENTO:	Fixed rates

FUENTE_NEGOCIO:	DIRECT SALES

NOMBRE_HOTEL:	Complejo Riviera Maya

NOMBRE_HABITACION:	SIG JUNIOR SUITE PS

VALOR_RESERVA:	4899.40

## Importación

In [ ]:
# Importar get_features desde src/funciones_produccion.py (ejecutar desde la carpeta notebooks/)

import sys
import os

module_path = os.path.abspath(os.path.join('..', 'src'))
if module_path not in sys.path:
    sys.path.append(module_path)

from funciones_produccion import get_features


## Parámetros

In [ ]:
# preparar los parámetros de la función

import pandas as pd

# Valores de ejemplo de una reserva
LLEGADA = pd.to_datetime('2023-11-06') # Convert to datetime
NOCHES = 17
PAX = 2
ADULTOS = 2
CLIENTE = 'AUREA TRAVEL CLUB_SOCIOS'
FECHA_TOMA = pd.to_datetime('2023-02-22 12:48:29') # Convert to datetime
FIDELIDAD = None
PAIS = 'ESTADOS UNIDOS'
SEGMENTO = 'Fixed rates'
FUENTE_NEGOCIO = 'DIRECT SALES'
NOMBRE_HOTEL = 'Complejo Riviera Maya'
NOMBRE_HABITACION = 'SIG JUNIOR SUITE PS'
VALOR_RESERVA = 4899.40

## get_features

In [ ]:
# LLamar a función get_features (devuelve un df)

new_reservation_df = get_features(
    LLEGADA, NOCHES, PAX, ADULTOS, CLIENTE, FECHA_TOMA, FIDELIDAD, PAIS, SEGMENTO, FUENTE_NEGOCIO, NOMBRE_HOTEL, NOMBRE_HABITACION, VALOR_RESERVA
)

display(new_reservation_df)

## Modelo

In [ ]:
# Cargar modelo (models/lightgbm.joblib, reentrenado con datos sintéticos)

import joblib
import json
import os

ruta_modelos = os.path.abspath(os.path.join('..', 'models'))
loaded_model = joblib.load(os.path.join(ruta_modelos, "lightgbm.joblib"))

with open(os.path.join(ruta_modelos, "metrics.json"), encoding="utf-8") as f:
    loaded_metadata = json.load(f)

print(f"Modelo {loaded_metadata['model_type']} cargado ({loaded_metadata['trained_on']}).")
print(f"AUC en test: {loaded_metadata['test_metrics']['auc_roc']:.4f}")
print(f"Variables utilizadas: {len(loaded_metadata['features'])}")


## Predicción

In [ ]:
new_reservation_df[loaded_model.feature_name_]

In [ ]:
# Hacer predicción de la reserva

# Identify categorical features from the model's feature names
categorical_features = [
    'FUENTE_NEGOCIO_SEGMENTO_CLIENTE',
    'PAIS_TOP_200',
    'COMPLEJO_RESERVA',
    'TIPO_VIAJERO',
    'HOTEL_HABITACION_TOP'
]

# Convert these columns in new_reservation_df to 'category' dtype
for col in categorical_features:
    if col in new_reservation_df.columns:
        new_reservation_df[col] = new_reservation_df[col].astype('category')

# Now, make the prediction
prediction_proba = loaded_model.predict_proba(new_reservation_df[loaded_model.feature_name_])
print(prediction_proba)

# Valores permitidos categóricas

A continuación están los valores posibles de las categorías de las variables:

CLIENTE

FIDELIDAD

PAIS

SEGMENTO

FUENTE_NEGOCIO

NOMBRE_HOTEL

NOMBRE_HABITACION

## CLIENTE

CLIENTE = 'AUREA TRAVEL CLUB_SOCIOS', 'WEB PROPIA', 'CALL CENTER', 'MOTOR RESERVAS GLOBAL',
       'OTA EUROPE BV', 'OTA AMERICAS INC', 'OTA LATAM S.A.', 'METABUSCADOR B2C',
       'TOUR OPERADOR NORTEAMERICA', 'TOUR OPERADOR ESPANA', 'AGENCIA MAYORISTA MX',
       'TOUR OPERADOR UK', 'AGENCIA CANADA', 'CUENTA CORPORATIVA A', 'CUENTA CORPORATIVA B', 'OTROS CANALES'

*(Lista anonimizada: los canales reales del cliente no se publican.)*


## FIDELIDAD

FIDELIDAD=None, 'Aurea Rewards', 'Partner Rewards', 'Aurea Connect'

## PAIS


PAIS='CANADA', 'ESTADOS UNIDOS', 'RUMANIA', 'ARGENTINA', 'SIN PAIS',
       'BRASIL', 'ALEMANIA', 'MEXICO', 'REINO UNIDO', 'COLOMBIA',
       'REPUBLICA DOMINICANA', 'SUECIA', 'REPUBLICA DE MOLDAVIA', 'CHILE',
       'ISLAS ULTRAMARINAS MENORES DE ESTADOS UNIDOS', 'ITALIA',
       'LUXEMBURGO', 'POLONIA', 'AFGANISTAN', 'URUGUAY', 'PERU',
       'FRANCIA', 'ESPAÑA', 'MAURITANIA', 'PUERTO RICO', 'DOMINICA',
       'PORTUGAL', 'COREA', 'DINAMARCA', 'CHINA', 'PARAGUAY', 'AUSTRALIA',
       'SUIZA', 'EMIRATOS ARABES UNIDOS', 'COREA DEL SUR', 'TURQUIA',
       'BELGICA', 'IRLANDA', 'JAMAICA', 'ECUADOR', 'CURAZAO', 'PANAMA',
       'JAPON', 'BOLIVIA', 'FEDERACION DE RUSIA', 'HONDURAS', 'RUSIA',
       'EL SALVADOR', 'MACAO', 'PAISES BAJOS', 'GRANADA', 'ISRAEL',
       'SINGAPUR', 'CUBA', 'ALBANIA', 'BULGARIA', 'SERBIA', 'ESLOVAQUIA',
       'VENEZUELA', 'INDIA', 'GUAM', 'HAITI', 'UCRANIA', 'COSTA RICA',
       'MACEDONIA', 'NIGERIA', 'AUSTRIA', 'MARRUECOS', 'FILIPINAS',
       'ISLAS VIRGENES BRITANICAS', 'ESTONIA', 'NORUEGA', 'GUAYANA',
       'TRINIDAD Y TOBAGO', 'HUNGRIA', 'BELICE', 'LITUANIA', 'GEORGIA',
       'CHIPRE', 'ARMENIA', 'EGIPTO', 'GUAYANA FRANCESA', 'BAHAMAS',
       'LIBANO', 'KUWAIT', 'GUATEMALA', 'VIETNAM', 'GIBRALTAR', 'ANDORRA',
       'NICARAGUA', 'NUEVA ZELANDA', 'BAHREIN', 'UZBEKISTAN', 'JORDANIA',
       'TAIWAN', 'PAKISTAN', 'REPUBLICA DE KOSOVO', 'ANGOLA', 'IRAN',
       'REPUBLICA CHECA', 'LETONIA', 'CABO VERDE', 'KENIA',
       'ISLAS VIRGENES DE EEUU', 'ANTARTIDA', 'BOSNIA Y HERCEGOVINA',
       'ARABIA SAUDI', 'IRAQ', 'REP. DEMOCRATICA DEL CONGO',
       'BIELORRUSIA', 'SUAZILANDIA', 'BERMUDAS', 'ESLOVENIA',
       'SAN CRISTOBAL Y NIEVES', 'MALAWI', 'ARGELIA', 'FINLANDIA',
       'SOMALIA', 'GABON', 'AZERBAIYAN', 'BARBADOS', 'OTROS', 'SRI LANKA',
       'SAMOA', 'TUNEZ', 'BANGLADES', 'MARTINICA', 'ISLA DE NAVIDAD',
       'MALASIA', 'CHAD', 'OMAN', 'SENEGAL', 'REUNION',
       'POLINESIA FRANCES', 'SUDAFRICA', 'GRECIA', 'MONACO',
       'LIECHTENSTEIN', 'ANGUILLA', 'ISLAS TURCAS Y CAICOS', 'TAILANDIA',
       'CROACIA', 'ISLANDIA', 'CAMERUN', 'NEPAL', 'KAZAJISTAN',
       'REPUBLICA DE MOZAMBIQUE', 'MYANMAR', 'REPUBLICA CENTROAFRICANA',
       'ETIOPIA', 'GHANA', 'BOTSWANA', 'SANTA LUCIA',
       'ISLAS ULTRAMARINAS MENORES DE ', 'MALTA', 'SIRIA',
       'ISLA DEL HOMBRE', 'MICRONESIA', 'ISLAS CAIMAN', 'ISLAS ALAND',
       'ZAMBIA', 'KIRGUISTAN', 'RUANDA', 'REPUBLICA DE MAURICIO', 'QATAR',
       'CIUDAD DEL VATICANO', 'ARUBA', 'TONGA', 'COSTA DE MARFIL',
       'HONG KONG', 'GUINEA', 'BRUNEI DARUSSALAM',
       'ARABE SAHARAUI (SAHARA OCCIDEN', 'MONTENEGRO', 'ARABIA SAUDITA',
       'BURKINA FASO', 'SAN MARTIN', 'BENIN', 'LAOS', 'INDONESIA',
       'PALESTINA', 'NETHERLANDS ANTILLES', 'MONGOLIA',
       'ANTIGUA Y BARBUDA', 'ISLAS VIRGENES DE LOS ESTADOS UNIDOS',
       'NIGER', 'REP DEMOCRATICA DEL CONGO', 'ISLAS VIRGENES DE EE.UU',
       'MAYOTTE', 'ZIMBABUE', 'SURINAM', 'GUERNSEY', 'ANGUILA',
       'TAYIKISTAN', 'TANZANIA', 'NAMIBIA', 'IR', 'BURUNDI',
       'GUADALUPE, SAINT MARTIN Y SAINT BARTHELEMY', 'UGANDA', 'FIYI'

## SEGMENTO

SEGMENTO='Fixed rates', 'Fixed Rates', 'BAR', 'Package', 'Loyalty',
       'Weddings', 'Group Leisure', 'MICE', 'Miscellaneous', 'Opaque',
       'Complimentary', 'Travel Industry'

## FUENTE_NEGOCIO

FUENTE_NEGOCIO='DIRECT SALES', 'T.O. / T.A.', 'E-COMMERCE', 'CORPORATE', 'OTHERS'

## NOMBRE_HOTEL

NOMBRE_HOTEL='Complejo Riviera Maya', 'Complejo Punta Cana','Complejo Costa Mujeres'

## NOMBRE_HABITACION

NOMBRE_HABITACION='MAYA_SIG - SIG JUNIOR SUITE PS', 'MAYA_COL - COL JUNIOR SUITE GV',
       'MAYA_WS - WS SUITE GARDEN VIEW',
       'MAYA_COL - COL DELUXE GARDEN VIEW',
       'CANA_PC - PC JUNIOR SUITE POOLSIDE',
       'CANA_PC - PC DELUXE POOLSIDE',
       'MAYA_SIG - SIG JS JACUZZI TERR PS',
       'MAYA_SIG - SIG JUNIOR SUITE GV',
       'MUJE_SIGC - SIGC JUNIOR SUITE GV',
       'CANA_BAV - BAV JUNIOR SUITE GV', 'CANA_BAV - BAV SUPERIOR JS GV',
       'CANA_PC - PC DELUXE GARDEN VIEW', 'MAYA_WS - WS JUNIOR SUITE BS',
       'MAYA_COL - COL ROMANCE VILLA SUI PS', 'MAYA_SIG - OTROS',
       'CANA_PC - PC JUNIOR SUITE GV', 'MUJE_CMU - CMU JUNIOR SUITE PS',
       'MUJE_SIG - SIG JUNIOR SUITE SW', 'CANA_SIG - OTROS',
       'CANA_PAL - PAL LOFT SUITE BS POV',
       'CANA_PAL - PAL DELUXE BEACHSIDE',
       'CANA_SIG - SIG JUNIOR SUITE SWIM UP',
       'CANA_SIG - SIG JUNIOR SUITE PS', 'MAYA_WS - WS JUNIOR SUITE GV',
       'MUJE_CMU - CMU JUNIOR SUITE PS OV',
       'MUJE_CMU - CMU JUNIOR SUITE GV',
       'MUJE_SIGC - SIGC JUNIOR SUITE BS OV',
       'MAYA_KAN - KAN ROMANCE VILLA SUI BS',
       'MUJE_SIG - SIG JUNIOR SUITE OV', 'MUJE_SIG - SIG JUNIOR SUITE GV',
       'CANA_SIG - SIG JUNIOR SUITE GV/PV',
       'CANA_PAL - PAL JUNIOR SUITE BS',
       'MAYA_KAN - KAN DELUXE GARDEN VIEW',
       'CANA_PAL - PAL JUNIOR SUITE GV',
       'MUJE_CMU_FS - CMU FS JUNIOR SUITE BS',
       'MAYA_SIG - SIG JUNIOR SUITE PS OV', 'CANA_PC - OTROS',
       'MAYA_KAN - KAN JUNIOR SUITE GV',
       'CANA_PAL - PAL JUNIOR SUITE SW BS', 'MUJE_SIGC - OTROS',
       'CANA_SIG - SIG JUNIOR SUITE PS OV',
       'MUJE_SIG - SIG JUNIOR SUITE BS OV', 'MUJE_CMU - OTROS',
       'MAYA_COL - COL JUNIOR SUITE PS',
       'CANA_SIG - SIG ROMANCE SUITE PS', 'CANA_PAL - PAL LOFT SUITE GV',
       'MUJE_CMU - CMU LOFT SUITE JT',
       'CANA_PAL - PAL DELUXE GARDEN VIEW', 'MUJE_CMU_FS - OTROS',
       'CANA_PAL - PAL JUNIOR SUITE BS OV', 'CANA_BAV - OTROS',
       'CANA_PAL - OTROS', 'MUJE_SIGC - SIGC JUNIOR SUITE SW',
       'MAYA_SIG - SIG ROMANCE BW BYTHE LAKE',
       'CANA_BAV - BAV PREMIUM JS GV', 'MUJE_SIGC - SIGC JUNIOR SUITE BS',
       'MUJE_SIGC - SIGC LOFT SUITE JT',
       'MUJE_SIGC - SIGC LOFT SUITE JT OV',
       'MAYA_SIG - SIG ST JACUZZI TERR PS', 'MUJE_SIG - OTROS',
       'MAYA_SIG - SIG JUNIOR SUITE PP PS', 'CANA_PC - PC LOFT SUITE GV',
       'CANA_BAV - BAV ROMANCE SUITE GV',
       'MUJE_CMU_FS - CMU FS JUNIOR SUI BS POV', 'MAYA_KAN - OTROS',
       'MAYA_COL - OTROS', 'MAYA_SIG - SIG JUNIOR SUITE PP GV',
       'MUJE_SIG - SIG LOFT SUITE JT OV', 'CANA_BAV - BAV ROOFTOP JT JS',
       'MAYA_WS - OTROS', 'MUJE_SIG - SIG LOFT SUITE JT',
       'MUJE_SIGC - SIGC JUNIOR SUITE OV'

In [ ]:
import pandas as pd

def get_prediccion(LLEGADA, NOCHES, PAX, ADULTOS, CLIENTE, FECHA_TOMA, FIDELIDAD, PAIS, SEGMENTO, FUENTE_NEGOCIO, NOMBRE_HOTEL, NOMBRE_HABITACION, VALOR_RESERVA):
    LLEGADA = pd.to_datetime(LLEGADA)
    FECHA_TOMA = pd.to_datetime(FECHA_TOMA)

    df_prediction = get_features(
        LLEGADA, NOCHES, PAX, ADULTOS, CLIENTE, FECHA_TOMA, FIDELIDAD, PAIS, SEGMENTO, FUENTE_NEGOCIO, NOMBRE_HOTEL, NOMBRE_HABITACION, VALOR_RESERVA
    )

    model_expected_features = list(loaded_model.feature_name_)
    df_for_prediction = df_prediction[model_expected_features].copy()

    categorical_cols_to_convert = [
        'FUENTE_NEGOCIO_SEGMENTO_CLIENTE',
        'PAIS_TOP_200',
        'COMPLEJO_RESERVA',
        'TIPO_VIAJERO',
        'HOTEL_HABITACION_TOP'
    ]

    for col in categorical_cols_to_convert:
        if col in df_for_prediction.columns:
            df_for_prediction[col] = df_for_prediction[col].astype('category')

    prediction_proba = loaded_model.predict_proba(df_for_prediction)
    return prediction_proba[0, 1]

In [ ]:
# Código para verificar en el notebook (mismos valores que en la app)
resultado = get_prediccion(
    LLEGADA='2026-03-11',
    NOCHES=7,
    PAX=2,
    ADULTOS=2,
    CLIENTE='MOTOR RESERVAS GLOBAL',
    FECHA_TOMA='2026-02-09 15:18:22',
    FIDELIDAD='Aurea Rewards',
    PAIS='ESPAÑA',
    SEGMENTO='Loyalty',
    FUENTE_NEGOCIO='DIRECT SALES',
    NOMBRE_HOTEL='Complejo Costa Mujeres',
    NOMBRE_HABITACION='CMU JUNIOR SUITE GV',
    VALOR_RESERVA=2626.06
)
print(f"Probabilidad de cancelación: {resultado:.2%}")

In [ ]:
# Código para verificar en el notebook (mismos valores que en la app)
resultado = get_prediccion(
    LLEGADA='2026-03-18',
    NOCHES=7,
    PAX=2,
    ADULTOS=2,
    CLIENTE='MOTOR RESERVAS GLOBAL',
    FECHA_TOMA='2026-02-16 15:24:40',
    FIDELIDAD=None,
    PAIS='ESPAÑA',
    SEGMENTO='BAR',
    FUENTE_NEGOCIO='DIRECT SALES',
    NOMBRE_HOTEL='Complejo Costa Mujeres',
    NOMBRE_HABITACION='CMU JUNIOR SUITE GV',
    VALOR_RESERVA=2602.11
)
print(f"Probabilidad de cancelación: {resultado:.2%}")